In [1]:
import json
import serial.tools.list_ports
from windfreak import SynthHD
import time
import pyvisa
from RsInstrument import RsInstrument, BinFloatFormat
import skrf as rf
import matplotlib.pyplot as plt
import numpy as np
from urllib.request import urlopen
from pathlib import Path
import os
import sys
from datetime import datetime
import matplotlib.colors as mcolors

In [2]:
ports = serial.tools.list_ports.comports()
for port in ports:
    print(f"Port Name:   {port.device}")
    print(f"Description: {port.description}")
    print(f"Hardware ID: {port.hwid}")  # Shows USB Vendor ID, Product ID, and Location string
    print("-" * 30)
    if 'A3E5' in port.hwid:
        print(f"Windfreak is on COM port {port.device}")
        synth = SynthHD(port.device)
        synth.init()

rf_probe = synth[0]
rf_probe.enable = False
rf_probe.frequency = 5.5e9   # set Channel A frequency
rf_probe.power = -30        # set Channel A power

rf_pump = synth[1]
rf_pump.enable = False
rf_pump.frequency = 8e9   # frequency in Hz
rf_pump.power = 0        #  power in dBm


rm = pyvisa.ResourceManager()
instruments = rm.list_resources()
noise_source_bias = None
for instrument in instruments:
    if '33220A' in instrument:
        # Initialize instrument
        noise_source_bias = rm.open_resource(instrument) 
        break
print("noise source:" + noise_source_bias.query('*IDN?').strip())


vna = None
for instrument in instruments:
    if 'Rohde' in instrument:
        # Initialize instrument
        vna = RsInstrument(instrument, id_query=True, reset=False) 
        break

print("vna:" + vna.query('*IDN?').strip())

spa = None
for instrument in instruments:
    if '7405' in instrument:
        spa = rm.open_resource(instrument) 
        break
print("spa:" + spa.query('*IDN?').strip())


qubit_flux_bias = None
for instrument in instruments:
    if 'YOKOGAWA' in instrument:
        # Initialize instrument
        qubit_flux_bias = rm.open_resource(instrument) 
        break
print("qubit_flux_bias:" + qubit_flux_bias.query('*IDN?').strip())

noise_diode_bias = None
for instrument in instruments:
    if 'GPIB_01_24431303637351D0D031' in instrument:
        # Initialize instrument
        noise_diode_bias = rm.open_resource(instrument) 
        break
print("noise_diode_bias:" + noise_diode_bias.query('*IDN?').strip())

WARM_SWITCHES_IP_ADDRESS = '169.254.12.12'
PROGRAMMABLE_ATTENUATOR_IP_ADDRESS = '169.254.11.11'

def send_url_command(ip_address, cmd_to_send):
    url = f"http://{ip_address}/:{cmd_to_send}"
    for attempt in range(3):
        try:
            with urlopen(url, timeout=3) as response:
                pte_return = response.read()
            if len(pte_return) > 100:
                print(f"Error, command not found: {url}")
                return "Invalid Command!"
            return pte_return.decode('utf-8').strip()
        except Exception as network_err:
            if attempt == 2:
                print(f"Hardware Network Warning: No response from {ip_address}. Error: {network_err}")
                return "No Response!"
            time.sleep(0.05)
            
def set_warm_switches_by_ip(ip_address, A, B, C, D):
    state_byte = 128 + int(A-1) + (2 * int(B-1)) + (4 * int(C-1)) + (8 * int(D-1))
    return send_url_command(ip_address, f"SETP={state_byte}")

def set_programmable_attenuator(attenuation):
    send_url_command(PROGRAMMABLE_ATTENUATOR_IP_ADDRESS, f"SETATT={attenuation}")

def get_switch_state(ip_address):
    raw_reply = send_url_command(ip_address, "SWPORT?")
    if any(err in raw_reply for err in ["Error", "No Response"]):
        return {"status": "Disconnected or Error", "raw_reply": raw_reply}
    try:
        state_byte = int(raw_reply.strip())
        working_value = state_byte - 128
        return {
            "status": "Online",
            "raw_byte": state_byte,
            "A": working_value % 2 + 1,
            "B": (working_value // 2) % 2 + 1,
            "C": (working_value // 4) % 2 + 1,
            "D": (working_value // 8) % 16 + 1
        }
    except ValueError:
        return {"status": f"Malformed hardware response: {raw_reply}"}

def get_attenuator_state(ip_address):
    # The direct URL that successfully delivers the information
    url = f"http://{ip_address}/ATT?"
    
    try:
        with urlopen(url, timeout=2.0) as response:
            # Read bytes, decode to string, and remove whitespace/newlines
            attenuation_value = response.read().decode('utf-8').strip()
            return float(attenuation_value)
            
    except URLError as e:
        print(f"Connection failed: {e.reason}")
        return None
    except ValueError:
        print(f"Parsing Error: Could not convert response to number: {attenuation_value}")
        return None
vna.go_to_local()


Port Name:   COM4
Description: USB Serial Device (COM4)
Hardware ID: USB VID:PID=0483:A3E5 SER=2068338D5232
------------------------------
Windfreak is on COM port COM4
Port Name:   COM3
Description: Intel(R) Active Management Technology - SOL (COM3)
Hardware ID: PCI\VEN_8086&DEV_02E3&SUBSYS_099F1028&REV_00\3&11583659&1&B3
------------------------------
noise source:Agilent Technologies,33220A,MY44053372,2.07-2.06-22-2
vna:Rohde&Schwarz,ZVA24-2Port,1145111024100131,3.30
spa:Hewlett-Packard, E7405A, SG45102671, A.14.04
qubit_flux_bias:YOKOGAWA,GS210,91U322681,2.02


VisaIOError: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.

In [3]:
instruments

('USB0::0x03EB::0x2065::YOKOGAWA_GS210_91U322681_2.02::INSTR',
 'USB0::0x03EB::0x2065::Rohde_Schwarz_ZVA24-2Port_1145111024100131_3.30::INSTR',
 'USB0::0x03EB::0x2065::Agilent_Technologies_33220A_MY44053372_2.07-2.06-2::INSTR',
 'USB0::0x03EB::0x2065::Hewlett-Packard__E7405A__SG45102671__A.14.04::INSTR',
 'USB0::0x03EB::0x2065::GPIB_01_24431303637351D0D031::INSTR',
 'ASRL3::INSTR',
 'ASRL4::INSTR')

In [ ]:
GPIB_01_24431303637351D0D031
GPIB_01_24431303637351D0D031